In [1]:
print("""
@File         : Performing mean or median imputation.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-25 15:32:27
@Email        : cuixuanstephen@gmail.com
@Description  : 执行平均值或中位数插补
""")


@File         : Performing mean or median imputation.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-25 15:32:27
@Email        : cuixuanstephen@gmail.com
@Description  : 执行平均值或中位数插补



> 如果变量呈正态分布，则使用均值插补，否则使用中位数插补。如果缺失数据的比例很高，则平均值和中位数插补可能会扭曲变量分布。

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from feature_engine.imputation import MeanMedianImputer

In [3]:
data = pd.read_csv('../../DATA/credit_approval_uci.csv')

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    data.drop('target', axis='columns'),
    data['target'], test_size=.3, random_state=0
)

In [5]:
numeric_vars = X_train.select_dtypes(exclude='O').columns.to_list()
numeric_vars

['A2', 'A3', 'A8', 'A11', 'A14', 'A15']

In [6]:
median_values = X_train[numeric_vars].median().to_dict()
median_values

{'A2': 28.835, 'A3': 3.0, 'A8': 1.29, 'A11': 0.0, 'A14': 160.0, 'A15': 6.0}

In [7]:
X_train_t = X_train.fillna(value=median_values)
X_test_t = X_test.fillna(value=median_values)

现在，让我们使用 scikit‑learn 用中位数填补缺失值。

In [8]:
imputer = SimpleImputer(strategy='median')

我们使用 `ColumnTransformer()` 将插补限制为数值变量：

In [9]:
ct = ColumnTransformer(
    [('imputer', imputer, numeric_vars)],
    remainder='passthrough',
    force_int_remainder_cols=False
).set_output(transform='pandas')

Scikit‑learn 可以返回 numpy 数组、pandas DataFrames 或 polar frame，具体取决于我们如何设置变换输出。默认情况下，它返回 numpy 数组。

In [10]:
ct.fit(X_train)

ColumnTransformer(force_int_remainder_cols=False, remainder='passthrough',
                  transformers=[('imputer', SimpleImputer(strategy='median'),
                                 ['A2', 'A3', 'A8', 'A11', 'A14', 'A15'])])

`ColumnTransformer()` 会改变输出中变量的名称。转换后的变量显示前缀 `imputer`，未改变的变量显示前缀 `remainder`。

In [13]:
ct.named_transformers_.imputer.statistics_

array([ 28.835,   3.   ,   1.29 ,   0.   , 160.   ,   6.   ])

In [14]:
X_train_t = ct.transform(X_train)
X_test_t = ct.transform(X_test)

In [15]:
X_train_t.head()

,imputer__A2,imputer__A3,imputer__A8,imputer__A11,imputer__A14,imputer__A15,remainder__A1,remainder__A4,remainder__A5,remainder__A6,remainder__A7,remainder__A9,remainder__A10,remainder__A12,remainder__A13
596,46.08,3.000,2.375,8.0,396.0,4159.0,a,u,g,c,v,t,t,t,g
303,15.92,3.000,1.290,0.0,120.0,0.0,a,u,g,q,v,NaN,NaN,f,g
204,36.33,2.125,0.085,1.0,50.0,1187.0,b,y,p,w,v,t,t,f,g
351,22.17,3.000,1.290,0.0,100.0,0.0,b,y,p,ff,ff,NaN,NaN,f,g
118,57.83,7.040,14.000,6.0,360.0,1332.0,b,u,g,m,v,t,t,t,g


In [17]:
X_train.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15
596,a,46.08,3.000,u,g,c,v,2.375,t,t,8,t,g,396.0,4159
303,a,15.92,NaN,u,g,q,v,NaN,NaN,NaN,0,f,g,120.0,0
204,b,36.33,2.125,y,p,w,v,0.085,t,t,1,f,g,50.0,1187
351,b,22.17,NaN,y,p,ff,ff,NaN,NaN,NaN,0,f,g,100.0,0
118,b,57.83,7.040,u,g,m,v,14.000,t,t,6,t,g,360.0,1332


最后，让我们使用 feature-engine 进行中值插补。

In [18]:
imputer = MeanMedianImputer(imputation_method='median', variables=numeric_vars)

默认情况下，`MeanMedianImputer()` 将插补 DataFrame 中的所有数值变量，忽略分类变量。使用变量参数将插补限制为数值变量的子集。

In [19]:
imputer.fit(X_train)

MeanMedianImputer(variables=['A2', 'A3', 'A8', 'A11', 'A14', 'A15'])

In [20]:
imputer.imputer_dict_

{'A2': 28.835, 'A3': 3.0, 'A8': 1.29, 'A11': 0.0, 'A14': 160.0, 'A15': 6.0}

In [21]:
X_train = imputer.transform(X_train)
y_test = imputer.transform(X_test)